# Identifying Trends Over Time Using Line Plots - Milestone

## Learning Objectives

By completing this milestone, you will:
- Understand what time-series data represents
- Visualize data changes over time using line plots
- Identify upward, downward, or stable trends
- Interpret patterns such as spikes or drops
- Build intuition for temporal analysis
- Detect anomalies or sudden shifts
- Use line plots as part of exploratory data analysis

## Why This Matters

Time-series analysis is critical because:
- **Time adds context** that static analysis cannot capture
- **Trends become visually clear** through temporal ordering
- **Patterns emerge** that help predict future behavior
- **Anomalies stand out** when viewed in temporal context
- **Better decisions** are made with historical perspective

> **Think of line plots as a story of how data evolves.**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

## 1. Understanding Time-Based Data

Time-series data has unique characteristics:

### Key Characteristics:

#### 1. **Temporal Ordering**
- Data points are ordered by time
- **Order matters!** Cannot be shuffled
- Past influences future (temporal dependency)

#### 2. **Time Intervals**
- **Regular**: Daily, weekly, monthly (evenly spaced)
- **Irregular**: Events, transactions (unevenly spaced)
- Important for analysis method selection

#### 3. **Common Patterns**
- **Trend**: Long-term increase or decrease
- **Seasonality**: Regular, predictable patterns
- **Cycles**: Irregular, longer-term fluctuations
- **Noise**: Random, unpredictable variation

#### 4. **Why Line Plots?**
- Show continuity between points
- Emphasize temporal flow
- Make trends visually obvious
- Highlight changes and patterns

### ⚠️ CRITICAL: Always ensure data is sorted by time before plotting!

## 2. Create Sample Time-Series Data

Let's create realistic time-series datasets with various patterns.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate daily dates for 2 years
start_date = datetime(2024, 1, 1)
dates = pd.date_range(start=start_date, periods=730, freq='D')

# 1. Sales data with upward trend and seasonality
trend = np.linspace(1000, 2500, 730)  # Upward trend
seasonality = 300 * np.sin(np.linspace(0, 4 * np.pi, 730))  # Seasonal pattern
noise = np.random.normal(0, 100, 730)  # Random noise
sales = trend + seasonality + noise

# Add some anomalies (promotional spikes and drops)
sales[180] += 800  # Big promotion
sales[365] += 600  # Holiday spike
sales[450] -= 400  # Supply chain issue
sales[600] += 700  # End of year sale

# 2. Website traffic with strong growth
base_traffic = np.linspace(5000, 25000, 730)
weekly_pattern = 2000 * np.sin(np.linspace(0, 104 * np.pi, 730))  # Weekly cycle
traffic_noise = np.random.normal(0, 500, 730)
traffic = base_traffic + weekly_pattern + traffic_noise
traffic = np.maximum(traffic, 0)  # Ensure no negative values

# 3. Temperature data with seasonal cycle
yearly_cycle = 20 * np.sin(np.linspace(0, 4 * np.pi, 730) - np.pi/2) + 20
daily_variation = np.random.normal(0, 3, 730)
temperature = yearly_cycle + daily_variation

# 4. Stock price with volatility
returns = np.random.normal(0.001, 0.02, 730)  # Daily returns
stock_price = 100 * np.exp(np.cumsum(returns))  # Compound returns

# 5. Customer signups with declining trend
declining_trend = np.linspace(500, 200, 730)
signup_noise = np.random.normal(0, 30, 730)
signups = declining_trend + signup_noise
signups = np.maximum(signups, 0)

# Create DataFrame
df = pd.DataFrame({
    'Date': dates,
    'Sales': sales,
    'Website_Traffic': traffic,
    'Temperature_C': temperature,
    'Stock_Price': stock_price,
    'Customer_Signups': signups
})

print(f"Dataset created with {len(df)} time points")
print(f"Time period: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")
print("\nFirst few rows:")
df.head(10)

In [ ]:
# Display basic info
print("Dataset Information:")
print(df.info())
print("\nDescriptive Statistics:")
df.describe()

## 3. Creating a Basic Line Plot

Let's start with a simple line plot to visualize one time series.

In [ ]:
# Ensure data is sorted by date (CRITICAL!)
df_sorted = df.sort_values('Date')

# Select column to plot
column = 'Sales'

# Create basic line plot
plt.figure(figsize=(14, 6))

plt.plot(df_sorted['Date'], df_sorted[column], 
         linewidth=2, color='steelblue', alpha=0.8, label='Actual Sales')

# Add trend line
x_numeric = np.arange(len(df_sorted))
z = np.polyfit(x_numeric, df_sorted[column], 1)
p = np.poly1d(z)
plt.plot(df_sorted['Date'], p(x_numeric), 
         '--', color='red', linewidth=2, alpha=0.7, label='Trend Line')

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Sales ($)', fontsize=12, fontweight='bold')
plt.title('Daily Sales Revenue Over Time', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate trend direction
if z[0] > 0:
    trend_direction = "UPWARD (increasing over time)"
elif z[0] < 0:
    trend_direction = "DOWNWARD (decreasing over time)"
else:
    trend_direction = "FLAT (stable over time)"

print(f"\nTrend Analysis:")
print(f"  Overall trend: {trend_direction}")
print(f"  Slope: {z[0]:.4f}")
print(f"  Average daily sales: ${df_sorted[column].mean():.2f}")

### Interpretation

From the plot above:
1. The **blue line** shows actual daily sales
2. The **red dashed line** shows the overall trend
3. **Fluctuations** around the trend represent seasonality and noise
4. **Visible spikes** may indicate special events or promotions

## 4. Identifying Trends with Moving Averages

Moving averages smooth out noise to reveal underlying trends.

In [ ]:
# Calculate moving averages
column = 'Website_Traffic'
df_sorted = df.sort_values('Date').copy()

df_sorted['MA_7'] = df_sorted[column].rolling(window=7, min_periods=1).mean()
df_sorted['MA_30'] = df_sorted[column].rolling(window=30, min_periods=1).mean()
df_sorted['MA_90'] = df_sorted[column].rolling(window=90, min_periods=1).mean()

# Plot with multiple moving averages
plt.figure(figsize=(14, 7))

plt.plot(df_sorted['Date'], df_sorted[column], 
         linewidth=1, color='gray', alpha=0.5, label='Original Data')
plt.plot(df_sorted['Date'], df_sorted['MA_7'], 
         linewidth=2, color='blue', alpha=0.7, label='7-Day Moving Average')
plt.plot(df_sorted['Date'], df_sorted['MA_30'], 
         linewidth=2, color='green', alpha=0.7, label='30-Day Moving Average')
plt.plot(df_sorted['Date'], df_sorted['MA_90'], 
         linewidth=2.5, color='red', alpha=0.8, label='90-Day Moving Average (Trend)')

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Website Traffic', fontsize=12, fontweight='bold')
plt.title('Website Traffic - Trend Analysis with Moving Averages', 
          fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("INTERPRETING MOVING AVERAGES")
print("=" * 80)
print("""
Moving averages smooth out short-term fluctuations:

- 7-Day MA:  Captures weekly patterns, still shows noise
- 30-Day MA: Monthly trend, reduces daily volatility
- 90-Day MA: Quarterly trend, reveals long-term direction

WHY USE MOVING AVERAGES?
✓ Remove noise to see the underlying trend
✓ Identify when trends change direction
✓ Compare short-term vs long-term behavior
✓ Make better decisions based on sustained patterns
""")

## 5. Detecting Anomalies

Anomalies are data points that deviate significantly from the norm.

In [ ]:
# Anomaly detection using Z-score method
column = 'Sales'
threshold = 2.5  # Standard deviations

df_sorted = df.sort_values('Date').copy()

# Calculate rolling statistics
df_sorted['Rolling_Mean'] = df_sorted[column].rolling(window=30, min_periods=1).mean()
df_sorted['Rolling_Std'] = df_sorted[column].rolling(window=30, min_periods=1).std()

# Identify anomalies
df_sorted['Z_Score'] = (df_sorted[column] - df_sorted['Rolling_Mean']) / df_sorted['Rolling_Std']
df_sorted['Is_Anomaly'] = (np.abs(df_sorted['Z_Score']) > threshold)

anomalies = df_sorted[df_sorted['Is_Anomaly']]

print(f"Anomaly Detection Results:")
print(f"  Total data points: {len(df_sorted)}")
print(f"  Anomalies found: {len(anomalies)}")
print(f"  Percentage: {(len(anomalies) / len(df_sorted) * 100):.2f}%")
print(f"\nAnomaly Details:")
print(anomalies[['Date', column, 'Z_Score']].to_string())

In [ ]:
# Visualize anomalies
plt.figure(figsize=(14, 7))

# Plot normal data
normal_data = df_sorted[~df_sorted['Is_Anomaly']]
plt.plot(normal_data['Date'], normal_data[column], 
         linewidth=2, color='steelblue', alpha=0.8, label='Normal Data')

# Plot rolling mean
plt.plot(df_sorted['Date'], df_sorted['Rolling_Mean'], 
         '--', linewidth=2, color='green', alpha=0.7, label='30-Day Moving Average')

# Highlight anomalies
if len(anomalies) > 0:
    plt.scatter(anomalies['Date'], anomalies[column], 
               color='red', s=100, zorder=5, label='Anomalies', marker='o')
    
    # Annotate anomalies
    for idx, row in anomalies.iterrows():
        plt.annotate(f"${row[column]:.0f}", 
                    xy=(row['Date'], row[column]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=9, color='red',
                    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
                    arrowprops=dict(arrowstyle='->', color='red'))

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Sales ($)', fontsize=12, fontweight='bold')
plt.title('Sales - Anomaly Detection', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Understanding Anomalies

Anomalies can indicate:

#### 1. **Special Events**
- Promotions, sales, marketing campaigns
- Holidays or seasonal events
- Product launches

#### 2. **Operational Issues**
- System outages or failures
- Supply chain disruptions
- Data collection errors

#### 3. **External Factors**
- Economic events
- Weather extremes
- Competitor actions

#### 4. **Natural Volatility**
- Random variation (not all anomalies are meaningful)
- Expected fluctuation in volatile systems

### ⚠️ ALWAYS INVESTIGATE:
- What happened on that date?
- Is there a reasonable explanation?
- Is this a one-time event or pattern?
- Should action be taken?

## 6. Comparing Multiple Time Series

Often we need to compare trends across different variables.

In [ ]:
# Normalized comparison (0-100 scale)
columns_to_compare = ['Sales', 'Website_Traffic', 'Customer_Signups']
df_sorted = df.sort_values('Date')

plt.figure(figsize=(14, 7))

for column in columns_to_compare:
    # Normalize to 0-100 scale
    normalized = 100 * (df_sorted[column] - df_sorted[column].min()) / \
                 (df_sorted[column].max() - df_sorted[column].min())
    plt.plot(df_sorted['Date'], normalized, 
            linewidth=2, label=column.replace('_', ' '), alpha=0.8)

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Normalized Value (0-100)', fontsize=12, fontweight='bold')
plt.title('Multiple Time Series - Normalized Comparison', 
          fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nComparison Insights:")
print("- Which metrics move together?")
print("- Which show opposite trends?")
print("- Which is most/least volatile?")

## 7. Analyzing Seasonality

Seasonal patterns repeat at regular intervals.

In [ ]:
# Analyze monthly and weekly patterns
column = 'Temperature_C'
df_sorted = df.sort_values('Date').copy()

# Extract time components
df_sorted['Month'] = df_sorted['Date'].dt.month
df_sorted['Month_Name'] = df_sorted['Date'].dt.strftime('%b')
df_sorted['DayOfWeek_Name'] = df_sorted['Date'].dt.strftime('%a')

# Monthly average pattern
monthly_avg = df_sorted.groupby('Month_Name')[column].mean().reindex(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)

# Plot monthly seasonality
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(df_sorted['Date'], df_sorted[column], linewidth=1.5, color='steelblue')
plt.title('Original Time Series', fontsize=12, fontweight='bold')
plt.xlabel('Date', fontsize=10)
plt.ylabel('Temperature (°C)', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(monthly_avg.index, monthly_avg.values, 
         marker='o', linewidth=2, color='green', markersize=8)
plt.title('Monthly Seasonal Pattern', fontsize=12, fontweight='bold')
plt.xlabel('Month', fontsize=10)
plt.ylabel('Average Temperature (°C)', fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSeasonality Insights:")
print(f"  Highest month: {monthly_avg.idxmax()} ({monthly_avg.max():.2f}°C)")
print(f"  Lowest month: {monthly_avg.idxmin()} ({monthly_avg.min():.2f}°C)")
print(f"  Seasonal variation: {monthly_avg.max() - monthly_avg.min():.2f}°C")

## 8. Practice Exercise: Your Turn!

Now apply what you've learned to your own data or a different column.

In [ ]:
# TODO: Load your own time-series data
# Example:
# my_df = pd.read_csv('../data/raw/your_timeseries_data.csv')
# my_df['Date'] = pd.to_datetime(my_df['Date'])

# TODO: Ensure data is sorted by date
# my_df = my_df.sort_values('Date')

# TODO: Create a line plot
# plt.figure(figsize=(14, 6))
# plt.plot(my_df['Date'], my_df['YourColumn'])
# plt.xlabel('Date')
# plt.ylabel('Your Metric')
# plt.title('Your Time Series Analysis')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

print("Complete the exercise by uncommenting and modifying the code above.")

## Summary and Key Takeaways

### What You Learned:
✓ What time-based data represents and why ordering matters  
✓ How to create clear line plots for temporal analysis  
✓ How to identify trends (upward, downward, stable)  
✓ How to detect and interpret anomalies  
✓ How to compare multiple time series  
✓ How to analyze seasonal patterns  

### Key Concepts:
1. **Always sort** time-series data by date before analysis
2. **Line plots** show continuity and flow over time
3. **Moving averages** help identify underlying trends
4. **Anomalies** require investigation, not automatic removal
5. **Normalize data** when comparing series with different scales
6. **Look for patterns**: trends, seasonality, cycles, anomalies
7. **Context matters** - numbers alone don't tell the story

### Best Practices:
- Label axes clearly with units and time periods
- Use appropriate time granularity (daily, weekly, monthly)
- Add trend lines to emphasize direction
- Annotate significant events or anomalies
- Consider multiple time scales (short-term vs long-term)
- Combine visualization with domain knowledge

### Next Steps:
- Apply line plots to your own time-series data
- Practice identifying trends and patterns
- Investigate anomalies with business context
- Combine with statistical analysis for deeper insights
- Record your 2-minute video walkthrough

### Video Walkthrough Requirements:
Your video should demonstrate:
1. Creating a line plot using time-based data
2. Explaining the observed trend
3. Pointing out notable changes or patterns
4. Explaining why line plots are suitable for time analysis

---

**Congratulations!** You've completed the Identifying Trends Over Time Milestone! 🎉